# Tech Challenge Fase 2
## 03.4 — Gold Indicadores

Consolida os produtos Gold de Alunos, Municípios e Estados em indicadores analíticos.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Consolidação municipal

In [0]:
bases = []

for ano in [2023, 2024, 2025]:
    df = ler_csv(
        GOLD_PATH
        / "municipios"
        / f"ano={ano}"
        / f"GOLD_MUNICIPIOS_{ano}.csv"
    )

    df["ANO"] = ano
    df["CO_MUNICIPIO"] = normalizar_codigo(
        df["CO_MUNICIPIO"]
    )

    if df.duplicated(
        subset=["ANO", "CO_MUNICIPIO"]
    ).any():
        raise ValueError(
            f"Indicadores {ano}: origem Gold Municípios duplicada."
        )

    bases.append(df)

df_indicadores = pd.concat(
    bases,
    ignore_index=True
)

df_indicadores["ano"] = df_indicadores["ANO"]
df_indicadores["id_municipio"] = (
    df_indicadores["CO_MUNICIPIO"]
    .astype(str)
)
df_indicadores["sigla_uf"] = (
    df_indicadores["SG_UF"]
    .astype(str)
)
df_indicadores["nome_municipio"] = (
    df_indicadores["NO_MUNICIPIO"]
    .astype(str)
)

if df_indicadores.duplicated(
    subset=["ano", "id_municipio"]
).any():
    raise ValueError(
        "GOLD_INDICADORES possui duplicidade por ano e município."
    )

print(
    "Gold Indicadores validada: "
    "uma linha por ano e município."
)

display(df_indicadores.head())

## 5. Persistência

In [0]:
salvar_csv(
    df_indicadores,
    GOLD_PATH / "indicadores",
    "GOLD_INDICADORES.csv"
)

print("Gold Indicadores salva com sucesso.")